# Loading OpenAI config and make simple text generation calls to chatgpt

In [1]:
import json
from dotenv import load_dotenv
from lib.messages import UserMessage, SystemMessage, ToolMessage  # Different message types
from lib.tooling import tool  # Tool decorator for creating AI tools
from lib.llm import LLM  # Our Language Model wrapper

In [2]:
load_dotenv()

True

In [3]:
chat_model = LLM()

In [4]:
# Method 1: Simple single-turn query
response = chat_model.invoke("What is an AI Agent?")
print("Single Query Response:\n", response.content)

Single Query Response:
 An AI agent is a system or entity that uses artificial intelligence techniques to perceive its environment, make decisions, and take actions to achieve specific goals. AI agents can operate autonomously or semi-autonomously and can be found in various applications, ranging from simple rule-based systems to complex machine learning models.

Key characteristics of AI agents include:

1. **Perception**: AI agents can gather information from their environment through sensors or data inputs. This could involve processing visual data, audio signals, or other forms of input.

2. **Decision-Making**: Based on the information they perceive, AI agents can analyze the data and make decisions. This can involve reasoning, planning, and learning from past experiences.

3. **Action**: After making decisions, AI agents can take actions to interact with their environment. This could involve physical actions (like a robot moving) or digital actions (like a software agent executin

In [ ]:
# Method 2: Multi-turn conversation with specific roles
messages = [
    SystemMessage(content="You're an OpenAI API specialist"),
    UserMessage(content="What is Function Calling?")
]
response = chat_model.invoke(messages)
print("\nStructured Conversation Response:\n", response.content)

# Building an AI Tool

In [ ]:
@tool
def get_weather(city: str):
    """Get the current temperature for a city.
    
    Args:
        city (str): Name of the city to check weather for
        
    Returns:
        dict: Contains temperature information for the requested city
    """
    # In a real application, this would call a weather API
    mock_weather = {
        "São Paulo": "28°C",
        "Oslo": "-3°C",
        "New York": "15°C",
        "Tokyo": "22°C"
    }
    return {"temperature": mock_weather.get(city, "Unknown")}

In [ ]:
# Bind the tool to an LLM
chat_model_with_tools = LLM(tools=[get_weather])

In [ ]:
# Set up our system with clear instructions
messages = [
    SystemMessage(
        content="You are a helpful assistant that can access a tool to get current temperature " 
                "for cities. Use the tool whenever someone asks about the weather or temperature " 
                "in a specific location. Infor the user if you don't know the answer."
    ),
    UserMessage(content="How cold is it in Oslo?")
]

In [ ]:
# AI decides to use the weather tool
ai_message = chat_model_with_tools.invoke(messages)
ai_message

In [ ]:
# Check messages structure
messages.append(ai_message)
messages

In [ ]:
# Extract the arguments
args = json.loads(messages[-1].tool_calls[0].function.arguments)
args

In [ ]:
# Execute the tool with the AI's requested parameters
tool_result = get_weather(**args)
tool_result

In [ ]:
# Create a tool response message
tool_message = ToolMessage(
    content=tool_result["temperature"], 
    tool_call_id=tool_call_id, 
    name="get_weather"
)
tool_message

In [ ]:
# Check messages structure
messages.append(tool_message)
messages

In [ ]:
# Let AI formulate final response
ai_message = chat_model_with_tools.invoke(messages)
ai_message

In [ ]:
# Check messages structure
messages.append(ai_message)
messages